# INF2006 - Big Data Project Team 18 Notebook

## How this notebook is organised

This is the team's notebook.

| Part | Task | What it does | What it saves |
|---|---|---|---|
| 1 | Task 2, data preparation (PySpark) | Cleans the prices, builds the weekly target, price and GDELT features, the chronological splits and the leakage checks | The prepared tables in the Hive database `HIVE_DB`, default `bigdata` (`train`, `val`, `test` and the others), when `WRITE_OUTPUTS = True` |
| 2 | Task 3, modelling (Spark ML) | Baseline, experiments, final model, saved PipelineModels | `pred_final_test`, `pred_gdelt_test`, the PipelineModels and a run summary |

**Run order on EMR**

1. Open the notebook with the PySpark kernel.
2. Run the `%%configure` cell below **first**.
3. In the Part 1 config set the HDFS paths and `WRITE_OUTPUTS = True`, then run Part 1 from top to bottom.
4. Run Part 2 from top to bottom. On the cluster it reads the tables `train`, `val` and `test` from `HIVE_DB` (default `bigdata`), which Part 1 saved.
5. When `OUT_S3` is set, Part 1 and Part 2 also copy the tables, the prediction tables, the models and the run summary to S3 (EMR only, a failed copy only prints a warning). If `OUT_S3 = None`, copy `hdfs:///project/models` and the evidence by hand before terminating the cluster. After, check the saved model in a fresh session with `reload_check.py`.

In [1]:
%%configure -f
{"driverMemory": "2g", "executorMemory": "6g", "executorCores": 2, "numExecutors":
2,
"conf": {"spark.dynamicAllocation.enabled": "false",
"spark.executor.memoryOverhead": "2g",
"spark.pyspark.python": "python3",
"spark.pyspark.virtualenv.enabled": "true",
"spark.pyspark.virtualenv.type": "native",
"spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"}}

## Part 1 – Task 2 (PySpark): price cleaning, target, features, GDELT features and chronological splits

**Inputs:** raw price CSV, Hive tables `trading_weeks` and `gdelt_daily` in `HIVE_DB` (default `bigdata`). **Outputs:** `clean_prices`, `rejected_prices`, `out_of_scope_prices`, `weekly_all`, `excluded_weeks`, `train`, `val`, `test`.

**Feature sets:** price-only = `FEATURE_COLS`, price+GDELT = `FEATURE_COLS + GDELT_COLS`, compared only on rows with `in_gdelt_window = true`.

Not features: `Name`, `date`, `week_friday`, `open`, `open_next_fri`, `label`, `pchg_next`, `pchg_day`, `split`, `next_anchor_date`, `in_gdelt_window`, `n_prior`, `hist_span_days`.

### 1.0 Config

All paths, dates and rules are set here. `GDELT_LAG_DAYS` is how many days before the prediction date the newest GDELT day may be. `MAX_HISTORY_SPAN_DAYS` is the longest calendar span allowed for the 21 prior trading rows (longer means a gap in the series). `RUN_CANARY` switches the leakage test on (evidence run) or off (timed runs). `WRITE_OUTPUTS` controls whether results are saved as Hive tables. Executor settings are in the `%%configure` cell, which must be run first.

In [2]:
RAW_PATH   = "hdfs:///project/raw/daily_price/all_stocks_5yr.csv" # HDFS path of the staged price CSV
END_DATE   = "2017-12-29" # last in-scope trading day
TRAIN_END  = "2015-12-31" # train: 2013-2015
VAL_END    = "2016-12-31" # validation: 2016
                          # test: 2017 (ends at END_DATE)
MIN_HISTORY = 21 # prior trading days needed for the longest feature
MAX_HISTORY_SPAN_DAYS = 34 # the 21 prior rows must span <= this many calendar days
GDELT_LAG_DAYS = 2 # newest GDELT day used = prediction date minus this many days
WRITE_OUTPUTS = True # first run False to test safely, then True to save the Hive tables
RUN_CANARY = True # True for the evidence run, False for timed runs
GDELT_PATH = None # Reads the Hive table gdelt_daily in HIVE_DB
HIVE_DB = "bigdata" # Hive database with B's tables and where Part 1 saves its tables: "bigdata" (B ran without RUN_ID) or e.g. "bigdata_run1" (B ran with RUN_ID=_run1)
OUT_S3 = "s3://big-date-team18-bucket/outputs"   # Second copy of the tables (and of Part 2's predictions, models and run summary) that outlives the cluster. None = no copy; use None for timed runs

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
import time
_T0, STAGE_TIMES = time.time(), []
def mark(label):
    STAGE_TIMES.append((label, round(time.time() - _T0, 1)))
    print(f"[{STAGE_TIMES[-1][1]:>8.1f}s] {label}")

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "16")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1794914433132_0004,pyspark,idle,":20888/proxy/application_1794914433132_0004/"">Link",":8042/node/containerlogs/container_1794914433132_0004_01_000001/livy"">Link",None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark 3.5.5-amzn-1 | shuffle partitions: 16

In [3]:
spark.sql("SHOW DATABASES").show(50, False)
spark.sql("SHOW TABLES IN default").show(50, False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+
|namespace|
+---------+
|bigdata  |
|default  |
+---------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+

In [4]:
spark.table(f"{HIVE_DB}.trading_weeks").printSchema()
spark.table(f"{HIVE_DB}.trading_weeks").show(5, False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- week_friday: date (nullable = true)
 |-- anchor_date: date (nullable = true)
 |-- next_anchor_date: date (nullable = true)
 |-- trading_days_in_week: long (nullable = true)
 |-- friday_was_holiday: boolean (nullable = true)
 |-- next_week_consecutive: boolean (nullable = true)
 |-- split: string (nullable = true)
 |-- label_known_in_split: boolean (nullable = true)
 |-- gdelt_split: string (nullable = true)
 |-- gdelt_label_known_in_split: boolean (nullable = true)
 |-- gdelt_prior_7d_covered: boolean (nullable = true)

+-----------+-----------+----------------+--------------------+------------------+---------------------+-----+--------------------+-----------+--------------------------+----------------------+
|week_friday|anchor_date|next_anchor_date|trading_days_in_week|friday_was_holiday|next_week_consecutive|split|label_known_in_split|gdelt_split|gdelt_label_known_in_split|gdelt_prior_7d_covered|
+-----------+-----------+----------------+--------------------+-------------

In [5]:
spark.table(f"{HIVE_DB}.gdelt_daily").printSchema()
spark.table(f"{HIVE_DB}.gdelt_daily").show(5, False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- day: date (nullable = true)
 |-- events: long (nullable = true)
 |-- articles: long (nullable = true)
 |-- avg_tone: double (nullable = true)
 |-- avg_goldstein: double (nullable = true)
 |-- neg_tone_share: double (nullable = true)
 |-- conflict_share: double (nullable = true)
 |-- bad_rows: long (nullable = true)

+----------+------+--------+-------------------+-------------------+------------------+-------------------+--------+
|day       |events|articles|avg_tone           |avg_goldstein      |neg_tone_share    |conflict_share     |bad_rows|
+----------+------+--------+-------------------+-------------------+------------------+-------------------+--------+
|2015-07-02|197720|3031170 |-2.0493683446237103|0.5221783329961562 |0.6779132106008496|0.14716771191584058|0       |
|2015-07-05|137463|2282152 |-1.9810724757643874|0.476600248794221  |0.6857263409062803|0.14529000531051992|0       |
|2015-07-08|234671|3931107 |-1.9349948436960682|0.6803243690102313 |0.6709819278905361|

### 1.1 Read raw prices

Reads the price CSV with an explicit schema. Lines Spark cannot parse are kept in `_corrupt`, not dropped. Each row keeps its source file (`src_file`) and original date text (`date_raw`) for provenance.

In [6]:
# An impossible date string (e.g. 2014-13-45) becomes null (-> rejected as missing_key) instead of raising a Spark upgrade exception
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
spark.conf.set("spark.sql.ansi.enabled", "false") # Spark 4.x: bad date text -> null, not an exception

schema = T.StructType([
    T.StructField("date", T.StringType()),
    T.StructField("open", T.DoubleType()),
    T.StructField("high", T.DoubleType()),
    T.StructField("low", T.DoubleType()),
    T.StructField("close", T.DoubleType()),
    T.StructField("volume", T.LongType()),
    T.StructField("Name", T.StringType()),
    T.StructField("_corrupt", T.StringType()),
])
raw = (spark.read.option("header", True).option("mode", "PERMISSIVE")
       .option("columnNameOfCorruptRecord", "_corrupt")
       .schema(schema).csv(RAW_PATH)
       .withColumn("src_file", F.input_file_name()) # provenance: which file the row came from
       .withColumn("date_raw", F.col("date")) # provenance: the original date text
       .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
       .cache())
raw_n = raw.count()
mark("read raw prices")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[    17.0s] read raw prices

### 1.2 Clean / rejected / out-of-scope

Every raw row lands in exactly one table, with the first failing rule as its reason: `corrupt_record`, `missing_key`, `missing_price`, `missing_volume`, `non_positive_price`, `high_below_low`, `price_outside_high_low`, `negative_volume`, `duplicate_key` (the bar with the highest volume is kept). Rows after `END_DATE` go to `out_of_scope_prices`. Zero-volume rows are flagged (`zero_volume`) but kept. Gaps are never filled.

**Reconciliation:** raw = clean + rejected + out_of_scope.

**Known limitation:** prices are not adjusted for some distributions and spin-offs, so a few one day jumps are artifacts. They are kept and counted in the evidence tables, not repaired.

In [7]:
price_cols = ["open", "high", "low", "close"]
any_null_price = F.greatest(*[F.col(c).isNull().cast("int") for c in price_cols]) == 1
any_nonpos = F.least(*[F.coalesce(F.col(c), F.lit(1.0)) for c in price_cols]) <= 0

stage1 = raw.withColumn(
    "reason",
    F.when(F.col("_corrupt").isNotNull(), "corrupt_record")
     .when(F.col("date").isNull() | F.col("Name").isNull(), "missing_key")
     .when(F.col("date") > F.lit(END_DATE).cast("date"), "OUT_OF_SCOPE")
     .when(any_null_price, "missing_price")
     .when(F.col("volume").isNull(), "missing_volume")
     .when(any_nonpos, "non_positive_price")
     .when(F.col("high") < F.col("low"), "high_below_low")
     .when((F.col("open") > F.col("high")) | (F.col("open") < F.col("low")) |
           (F.col("close") > F.col("high")) | (F.col("close") < F.col("low")),
           "price_outside_high_low")
     .when(F.col("volume") < 0, "negative_volume"))

# duplicates (same Name + date) among otherwise valid rows: keep one, reject the rest
w_dup = Window.partitionBy("Name", "date").orderBy(F.col("volume").desc())
valid = stage1.filter(F.col("reason").isNull()).withColumn("rn", F.row_number().over(w_dup))
dups = valid.filter("rn > 1").withColumn("reason", F.lit("duplicate_key")).drop("rn")
clean = (valid.filter("rn = 1").drop("rn", "reason", "_corrupt", "date_raw", "src_file")
         .withColumn("zero_volume", (F.col("volume") == 0))) # flagged, not removed
clean = clean.cache()

out_of_scope = stage1.filter(F.col("reason") == "OUT_OF_SCOPE").drop("reason")
rejected = (stage1.filter(F.col("reason").isNotNull() & (F.col("reason") != "OUT_OF_SCOPE"))
            .unionByName(dups.select(stage1.columns)))

clean_n, rej_n, oos_n = clean.count(), rejected.count(), out_of_scope.count()
print(f"RAW={raw_n}  CLEAN={clean_n}  REJECTED={rej_n}  OUT_OF_SCOPE={oos_n}")
assert rej_n / raw_n < 0.01, (
    f"{rej_n} of {raw_n} price rows rejected. Check the staged file: dates must be "
    "yyyy-MM-dd and the file must be the original, not the Excel re-saved copy.")
assert raw_n == clean_n + rej_n + oos_n, "row-count reconciliation FAILED"
rejected.groupBy("reason").count().orderBy(F.desc("count")).show(truncate=False)
mark("clean / reject / out-of-scope")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

RAW=605910  CLEAN=605887  REJECTED=23  OUT_OF_SCOPE=0
+----------------------+-----+
|reason                |count|
+----------------------+-----+
|price_outside_high_low|11   |
|missing_price         |11   |
|high_below_low        |1    |
+----------------------+-----+

[    28.6s] clean / reject / out-of-scope

In [8]:
# Cross-check with Person B (comparison only, nothing is changed): the price rows B's Hadoop Streaming job rejected
# (Hive table price_rejects, columns ticker and trade_date) should be the same (Name, date) rows as `rejected` above.
# C's rejected rows are the official ones; B's table is the independent second count.
try:
    b_rej = (spark.table(f"{HIVE_DB}.price_rejects")
                  .select(F.col("ticker").alias("Name"), F.col("trade_date").cast("date").alias("date"))
                  .distinct())
    b_rej.count()                                              # forces the read, so a missing table is caught here
except Exception as e:   # noqa: BLE001  local dry run, or B's tables are not in HIVE_DB
    b_rej = None
    print(f"cross-check with B skipped: {HIVE_DB}.price_rejects could not be read ({type(e).__name__})")

if b_rej is not None:
    c_rej = rejected.select("Name", "date").distinct()
    only_in_c = c_rej.join(b_rej, ["Name", "date"], "left_anti")
    only_in_b = b_rej.join(c_rej, ["Name", "date"], "left_anti")
    n_c, n_b, n_only_c, n_only_b = c_rej.count(), b_rej.count(), only_in_c.count(), only_in_b.count()
    print(f"C rejected: {n_c}   B price_rejects: {n_b}   only in C: {n_only_c}   only in B: {n_only_b}")
    if n_only_c or n_only_b:   # report, do not stop the run: a difference is a finding to explain, not a reason to lose the cluster time
        print("MISMATCH between C and B; rows that differ:")
        only_in_c.show(20, truncate=False)
        only_in_b.show(20, truncate=False)
    else:
        print("MATCH: C and B rejected exactly the same rows")
mark("cross-check rejected rows with B")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

C rejected: 23   B price_rejects: 23   only in C: 0   only in B: 0
MATCH: C and B rejected exactly the same rows
[    37.7s] cross-check rejected rows with B

### 1.3 Price features

Six features for each stock: `pchg_1d`, `pchg_5d`, `pchg_20d`, `vol_ratio_5_20`, `volatility_20d`, `range_prev`. All windows end at the previous trading row, so nothing from the prediction day is used. `pchg_day` (the same-day return) is a helper column and never a feature. `n_prior` counts prior rows and `hist_span_days` measures how many calendar days they cover, so windows that cross a gap can be excluded.

In [9]:
# Price features for every row. Each uses prior trading days only (windows end at -1, never include today)
def add_price_features(df):
    w  = Window.partitionBy("Name").orderBy("date")
    w5, w20 = w.rowsBetween(-5, -1), w.rowsBetween(-20, -1)
    wN = w.rowsBetween(-MIN_HISTORY, -1)
    x = df.withColumn("pchg_day", F.col("close") / F.lag("close", 1).over(w) - 1)
    return (x
         .withColumn("pchg_1d",  F.lag("pchg_day", 1).over(w))
         .withColumn("pchg_5d",  F.lag("close", 1).over(w) / F.lag("close", 6).over(w) - 1)
         .withColumn("pchg_20d", F.lag("close", 1).over(w) / F.lag("close", 21).over(w) - 1)
         .withColumn("vol_avg5",  F.avg("volume").over(w5))
         .withColumn("vol_avg20", F.avg("volume").over(w20))
         .withColumn("vol_ratio_5_20",
                     F.when(F.col("vol_avg20") > 0, F.col("vol_avg5") / F.col("vol_avg20")))
         .withColumn("volatility_20d", F.stddev_samp("pchg_day").over(w20))
         .withColumn("range_prev",
                     (F.lag("high", 1).over(w) - F.lag("low", 1).over(w)) / F.lag("close", 1).over(w))
         .withColumn("n_prior", F.count("close").over(wN))
         .withColumn("hist_span_days", F.datediff("date", F.lag("date", MIN_HISTORY).over(w))))

d = add_price_features(clean)

FEATURE_COLS = ["pchg_1d", "pchg_5d", "pchg_20d", "vol_ratio_5_20", "volatility_20d", "range_prev"]
# NOT features (identifiers / target-related): Name, date, week_friday, open, open_next_fri, label, pchg_next, pchg_day, split, next_anchor_date, in_gdelt_window, n_prior, hist_span_days

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### 1.4 Trading-week calendar

A week ends on Friday. `anchor_date` is the last trading day on or before that Friday (a holiday Friday uses the previous trading day). A trading day needs at least 250 of the 505 series. The calendar is derived from the clean prices, then cross-checked against the Hive table `bigdata.trading_weeks`. Each week gets its split (train 2013-2015, validation 2016, test 2017) and a flag showing whether its outcome falls inside the same split.

In [10]:
# Rule: a week ends on Friday, anchor_date = last trading day on or before that Friday,
# so a holiday Friday uses the previous trading day. Label = open(next anchor) > open(this anchor)

# One definition of the chronological split, driven by TRAIN_END / VAL_END (no hardcoded years)
TRAIN_END_D, VAL_END_D = F.lit(TRAIN_END).cast("date"), F.lit(VAL_END).cast("date")
def split_of(c):
    return F.when(c <= TRAIN_END_D, "train").when(c <= VAL_END_D, "validation").otherwise("test")

n_series = clean.select("Name").distinct().count()
MIN_SERIES = 250 if n_series >= 500 else max(1, n_series // 2) # B: a trading day needs >=250 of 505 series

trading_days = (clean.filter(F.col("date").between("2013-02-08", END_DATE))
                .groupBy("date").agg(F.countDistinct("Name").alias("n_series"))
                .filter(F.col("n_series") >= MIN_SERIES))
week_friday = F.expr("date_add(date '1970-01-02', "
                     "cast(ceil(datediff(`date`, date '1970-01-02') / 7.0) as int) * 7)") # 1970-01-02 was a Friday
weeks = (trading_days.withColumn("week_friday", week_friday)
         .groupBy("week_friday")
         .agg(F.max("date").alias("anchor_date"), F.count("*").alias("trading_days_in_week")))
wk = Window.orderBy("week_friday") # tiny table (~260 rows), a single partition is fine
cal = (weeks
       .withColumn("next_week_friday", F.lead("week_friday").over(wk))
       .withColumn("next_anchor_date", F.lead("anchor_date").over(wk))
       .withColumn("friday_was_holiday", F.col("anchor_date") != F.col("week_friday"))
       .withColumn("next_week_consecutive", F.datediff("next_week_friday", "week_friday") == 7)
       .withColumn("split", split_of(F.col("anchor_date")))
       .withColumn("label_known_in_split", # the outcome must fall inside the same split
                   F.col("next_anchor_date").isNotNull() &
                   (split_of(F.col("next_anchor_date")) == split_of(F.col("anchor_date")))))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
HIVE_ANCHOR_COL = "anchor_date"
b_cal = (spark.table(f"{HIVE_DB}.trading_weeks")
         .select(F.col(HIVE_ANCHOR_COL).cast("date").alias("anchor_date"))
         .filter(F.col("anchor_date").between("2013-02-08", END_DATE)))
only_hive  = b_cal.exceptAll(cal.select("anchor_date")).count()
only_spark = cal.select("anchor_date").exceptAll(b_cal).count()
print(f"Calendar cross-check: weeks in Hive={b_cal.count()}  weeks in Spark={cal.count()}  "
      f"only in Hive={only_hive}  only in Spark={only_spark}")
assert only_hive == 0 and only_spark == 0, "Hive trading_weeks and this notebook's calendar differ: show the differing dates to B before going on"


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Calendar cross-check: weeks in Hive=256  weeks in Spark=256  only in Hive=0  only in Spark=0

### 1.5 GDELT weekly features

Market-wide features from `bigdata.gdelt_daily` (keyed on `DATEADDED`, the day GDELT added the records). For a week anchored on date A, only days A-(L+13) to A-L are used, where L = `GDELT_LAG_DAYS`: the newest 7 days (level, tone, negative and conflict share) and the 7 days before (for the change in volume). A week gets features only when both windows are complete. There is no filling.

In [12]:
# gdelt_daily.day = date GDELT added the records (DATEADDED). For a week anchored on A, only days
# A-(L+13) ... A-L are used (L = GDELT_LAG_DAYS), so nothing from the last L days before the prediction date, or later, can leak in
GDELT_COLS = ["g_events_7d", "g_events_chg", "g_tone_7d", "g_neg_share_7d", "g_conflict_share_7d"]
if GDELT_PATH: # local development: CSV export of the Hive table
    g_schema = T.StructType([
        T.StructField("day", T.StringType()), T.StructField("events", T.LongType()),
        T.StructField("articles", T.LongType()), T.StructField("avg_tone", T.DoubleType()),
        T.StructField("avg_goldstein", T.DoubleType()), T.StructField("neg_tone_share", T.DoubleType()),
        T.StructField("conflict_share", T.DoubleType()), T.StructField("bad_rows", T.LongType())])
    gd = spark.read.option("header", True).schema(g_schema).csv(GDELT_PATH)
else: # EMR: Hive table built (02_gdelt_daily.hql)
    gd = spark.table(f"{HIVE_DB}.gdelt_daily")
    need = {"day", "events", "avg_tone", "neg_tone_share", "conflict_share"}
    assert need <= set(gd.columns), f"gdelt_daily has columns {gd.columns}"
# DATEADDED day: accept both 2015-08-15 and 20150815 formats; fail loudly if any day cannot be parsed
gd = gd.withColumn("day", F.coalesce(F.to_date(F.col("day").cast("string"), "yyyy-MM-dd"),
                                     F.to_date(F.col("day").cast("string"), "yyyyMMdd")))
assert gd.filter(F.col("day").isNull()).count() == 0, "gdelt_daily.day could not be parsed as a date"
gd = (gd.filter(F.col("events") > 0)
      .withColumn("tone_w", F.col("avg_tone") * F.col("events")) # event-weighted sums
      .withColumn("neg_w",  F.col("neg_tone_share") * F.col("events"))
      .withColumn("conf_w", F.col("conflict_share") * F.col("events")))

L = GDELT_LAG_DAYS # newest usable GDELT day is anchor - L
win = (cal.select("anchor_date")
       .join(gd, (F.col("day") >= F.date_sub("anchor_date", L + 13)) & (F.col("day") <= F.date_sub("anchor_date", L)))
       .withColumn("cur", F.col("day") >= F.date_sub("anchor_date", L + 6))) # newest 7 usable days vs the 7 before
cur, prev = F.col("cur"), ~F.col("cur")
g_week = (win.groupBy("anchor_date").agg(
    F.sum(F.when(cur, 1).otherwise(0)).alias("days_cur"), F.sum(F.when(prev, 1).otherwise(0)).alias("days_prev"),
    F.sum(F.when(cur, F.col("events"))).alias("ev_cur"), F.sum(F.when(prev, F.col("events"))).alias("ev_prev"),
    F.sum(F.when(cur, F.col("tone_w"))).alias("tone_cur"), F.sum(F.when(cur, F.col("neg_w"))).alias("neg_cur"),
    F.sum(F.when(cur, F.col("conf_w"))).alias("conf_cur")))
complete = (F.col("days_cur") == 7) & (F.col("days_prev") == 7) # no partial windows, no filling
g_week = g_week.select(
    "anchor_date",
    F.when(complete, F.col("ev_cur")).alias("g_events_7d"),
    F.when(complete, F.col("ev_cur") / F.col("ev_prev") - 1).alias("g_events_chg"), # week-on-week change
    F.when(complete, F.col("tone_cur") / F.col("ev_cur")).alias("g_tone_7d"), # <0 = mostly negative
    F.when(complete, F.col("neg_cur") / F.col("ev_cur")).alias("g_neg_share_7d"),
    F.when(complete, F.col("conf_cur") / F.col("ev_cur")).alias("g_conflict_share_7d"))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### 1.6 Weekly observations, target and exclusions

One row per stock per week. The target is `label` = 1 if the open on the next anchor date is above the open on this anchor date, plus `pchg_next` as the numeric change. The target is filled only when its outcome falls in the same split and the next week directly follows. Rows that cannot be used are kept in `excluded_weeks` with one reason: `no_price_on_anchor`, `insufficient_history`, `gappy_history`, `null_feature`, `no_next_week`, `calendar_gap_after_week`, `target_crosses_split_boundary`, `no_price_on_next_anchor`. `in_gdelt_window` marks rows with complete GDELT features, the subset for comparing price-only against price+GDELT.

In [13]:
# Grid = every calendar week inside a series' own life span, so a missing price is reported, not silently lost
rng  = clean.groupBy("Name").agg(F.min("date").alias("first_d"), F.max("date").alias("last_d"))
grid = (cal.crossJoin(rng)
        .filter((F.col("anchor_date") >= F.col("first_d")) & (F.col("anchor_date") <= F.col("last_d"))))
at_anchor = d.select("Name", F.col("date").alias("anchor_date"), "open", "n_prior", "hist_span_days", *FEATURE_COLS)
nxt = clean.select("Name", F.col("date").alias("next_anchor_date"), F.col("open").alias("open_next_fri"))

# The label is only filled when its full outcome lies inside the same split and the next week directly
# follows this one, otherwise it stays null, so boundary rows in weekly_all never carry a future outcome
known = (F.col("label_known_in_split")
         & F.coalesce(F.col("next_week_consecutive"), F.lit(False))
         & F.col("open_next_fri").isNotNull())

weekly = (grid.join(at_anchor, ["Name", "anchor_date"], "left")
          .join(nxt, ["Name", "next_anchor_date"], "left")
          .join(g_week, "anchor_date", "left") # market-wide GDELT features (null outside coverage)
          .withColumn("in_gdelt_window", # rows usable for the price-only vs price+GDELT comparison
                      F.greatest(*[F.col(x).isNull().cast("int") for x in GDELT_COLS]) == 0)
          .withColumn("date", F.col("anchor_date"))
          .withColumn("label", F.when(known, (F.col("open_next_fri") > F.col("open")).cast("int")))
          .withColumn("pchg_next", # numeric target for regression / error analysis, never a feature
                      F.when(F.col("label").isNotNull(), F.col("open_next_fri") / F.col("open") - 1))
          .withColumn("exclusion_reason",
              F.when(F.col("open").isNull(), "no_price_on_anchor")
               .when(F.col("n_prior") < MIN_HISTORY, "insufficient_history")
               .when(F.col("hist_span_days") > MAX_HISTORY_SPAN_DAYS, "gappy_history")
               .when(F.greatest(*[F.col(c).isNull().cast("int") for c in FEATURE_COLS]) == 1, "null_feature")
               .when(F.col("next_anchor_date").isNull(), "no_next_week")
               .when(~F.col("next_week_consecutive"), "calendar_gap_after_week")
               .when(~F.col("label_known_in_split"), "target_crosses_split_boundary")
               .when(F.col("open_next_fri").isNull(), "no_price_on_next_anchor")))
weekly = weekly.cache() # reused by the checks, evidence tables and writes

model_rows = weekly.filter(F.col("exclusion_reason").isNull())
excluded   = weekly.filter(F.col("exclusion_reason").isNotNull())
keep = ["Name", "date", "week_friday", "open", "open_next_fri", "label", "pchg_next", "split",
        "next_anchor_date", "in_gdelt_window"] + FEATURE_COLS + GDELT_COLS
train, val, test = [model_rows.filter(F.col("split") == s).select(keep)
                    for s in ("train", "validation", "test")]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### 1.7 Leakage and integrity checks

Assertions fail the run if any of these happen:
- rows do not reconcile
- a target date leaves its split
- train, validation and test overlap in time
- a (stock, date) pair repeats
- a model row has no target
- an identifier or target column is used as a feature
- a GDELT window reaches outside its coverage
- a recomputed GDELT value differs

The leakage canary (`RUN_CANARY`) multiplies all prices and volumes from one date onward and recomputes the features. Features on that date must not change. The same-day return `pchg_day` must change, which proves the test can detect leakage.

In [14]:
split_end = {"train": TRAIN_END, "validation": VAL_END, "test": END_DATE}

# row-count reconciliation
weekly_n, model_n, excl_n = weekly.count(), model_rows.count(), excluded.count()
split_n = {r["split"]: r["count"] for r in model_rows.groupBy("split").count().collect()}
assert weekly_n == model_n + excl_n, "weekly rows do not reconcile with model_rows + excluded"
assert model_n == sum(split_n.values()), "train + val + test != model_rows"

# chronological splits: targets stay inside their period, periods do not overlap
bounds = {r["split"]: r for r in model_rows.groupBy("split").agg(
    F.min("date").alias("min_d"), F.max("date").alias("max_d"),
    F.max("next_anchor_date").alias("max_target_d")).collect()}
for s_name, r in bounds.items():
    assert str(r["max_target_d"]) <= split_end[s_name], f"{s_name}: target date beyond its own period"
assert str(bounds["train"]["max_d"]) <= TRAIN_END < str(bounds["validation"]["min_d"]), "train/validation overlap"
assert str(bounds["validation"]["max_d"]) <= VAL_END < str(bounds["test"]["min_d"]), "validation/test overlap"

# row-level integrity
assert model_rows.groupBy("Name", "date").count().filter("count > 1").count() == 0, "duplicate (Name, date)"
assert model_rows.filter(~F.dayofweek("date").between(2, 6)).count() == 0, "anchor is not a weekday"
assert model_rows.filter(F.col("label").isNull() | F.col("pchg_next").isNull()).count() == 0, "a model row has no target"
assert weekly.filter("exclusion_reason = 'target_crosses_split_boundary' AND label IS NOT NULL").count() == 0, \
    "a boundary row still carries a label"

# identifiers and target-related columns must never be features
NOT_FEATURES = {"Name", "date", "week_friday", "open", "open_next_fri", "label", "pchg_next", "pchg_day",
                "split", "next_anchor_date", "in_gdelt_window", "n_prior", "hist_span_days"}
assert not set(FEATURE_COLS + GDELT_COLS) & NOT_FEATURES, "an identifier / target column is used as a feature"

# GDELT windows stay inside the coverage and respect the lag
gd_rows = model_rows.filter("in_gdelt_window")
if gd_rows.take(1):
    g_min, g_max = gd.agg(F.min("day"), F.max("day")).first() # actual GDELT coverage
    assert gd_rows.filter((F.date_sub("date", GDELT_LAG_DAYS + 13) < F.lit(g_min)) |
                          (F.date_sub("date", GDELT_LAG_DAYS) > F.lit(g_max))).count() == 0, \
        "a GDELT window reaches outside the coverage or closer to the anchor than the lag allows"

    # independent recomputation of one window: g_events_7d = sum of events over anchor-(L+6) ... anchor-L
    a = gd_rows.orderBy("date").select("date").first()["date"]
    exp = (gd.filter((F.col("day") >= F.date_sub(F.lit(a), GDELT_LAG_DAYS + 6)) &
                     (F.col("day") <= F.date_sub(F.lit(a), GDELT_LAG_DAYS)))
             .agg(F.sum("events")).first()[0])
    got = gd_rows.filter(F.col("date") == a).select("g_events_7d").first()[0]
    assert exp == got, f"g_events_7d spot check failed at {a}: {got} != {exp}"
    print("GDELT spot check OK at", a)
else:
    print("WARNING: no model rows have a complete GDELT window (check GDELT coverage / schema)")

# leakage canary: changing prices/volume on or after date T must not change any feature at T
if RUN_CANARY:
    anchors = sorted(r[0] for r in cal.select("anchor_date").collect()) # ~260 calendar rows only
    canary = anchors[len(anchors) // 2]
    # only rows within +/- 90 days of the canary date can matter for a feature at that date,
    # so test on that slice: same invariance claim, a fraction of the compute
    base = clean.filter(F.col("date").between(F.date_sub(F.lit(canary), 90), F.date_add(F.lit(canary), 90)))
    after = F.col("date") >= F.lit(canary)
    pert = base
    for c in price_cols:
        pert = pert.withColumn(c, F.col(c) * F.when(after, 3.0).otherwise(1.0))
    pert = pert.withColumn("volume", (F.col("volume") * F.when(after, 5).otherwise(1)).cast("long"))
    cols = FEATURE_COLS + ["pchg_day"]
    both = (add_price_features(base).filter(F.col("date") == F.lit(canary)).select("Name", *cols)
            .unionByName(add_price_features(pert).filter(F.col("date") == F.lit(canary)).select("Name", *cols)))
    agg = both.groupBy("Name").agg(*[(F.max(c) - F.min(c)).alias(c) for c in cols],
                                   *[F.count(c).alias(c + "_n") for c in cols])
    diff = agg.agg(*[F.max(F.coalesce(F.col(c), F.lit(0.0))).alias(c) for c in cols]).first().asDict()
    assert max(diff[c] for c in FEATURE_COLS) < 1e-9, f"LEAKAGE: {diff}"
    assert diff["pchg_day"] > 1e-6, "positive control did not react: the test is blind"
    assert agg.filter(" OR ".join(f"{c}_n = 1" for c in FEATURE_COLS)).count() == 0, "null/non-null mismatch"
    print("canary OK at", canary)
else:
    print("canary skipped (RUN_CANARY = FALSE)")

print("All integrity checks passed.")
mark("weekly table + integrity checks")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

GDELT spot check OK at 2015-07-17
canary OK at 2015-07-24
All integrity checks passed.
[    72.8s] weekly table + integrity checks

### 1.8 Evidence tables

Run IDs (Spark version, application ID), row-count reconciliations, quality flags, weeks per split, rows and class balance per split, rows with complete GDELT features, exclusions by split and reason, gaps found inside series and the output schema.

In [15]:
# run identification
import datetime, sys
print("Spark version  :", spark.version)
print("Application ID :", spark.sparkContext.applicationId)
print("Run date (UTC) :", datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"))
print("Python version :", sys.version.split()[0])
print()

# row-count reconciliation: source vs prepared
weekly_n, model_n, excl_n = weekly.count(), model_rows.count(), excluded.count()
split_n = {r["split"]: r["count"] for r in model_rows.groupBy("split").count().collect()}
print("Row-count reconciliation:")
for name_, left_, right_ in [
        ("price rows : raw = clean + rejected + out_of_scope", raw_n, clean_n + rej_n + oos_n),
        ("weekly rows: weekly_all = model_rows + excluded",    weekly_n, model_n + excl_n),
        ("model rows : model_rows = train + val + test",       model_n, sum(split_n.values()))]:
    print(f"  {name_:52s} {left_:>9} vs {right_:>9}   {'OK' if left_ == right_ else 'MISMATCH'}")

# quality flags: kept in the data, reported here
print("\nQuality flags (rows are kept, not removed):")
print("  zero-volume rows in clean_prices          :", clean.filter("zero_volume").count())
print("  rows with |daily price change| > 50% (check for unadjusted splits):", d.filter(F.abs("pchg_day") > 0.5).count())
d.filter(F.abs("pchg_day") > 0.5).select("Name", "date", "close", "pchg_day").orderBy("date").show(10, truncate=False)

# calendar, splits, GDELT window, exclusions
print("\nCalendar per split (compare with 04_checks.hql output):")
(cal.groupBy("split").agg(F.count("*").alias("weeks"),
     F.sum(F.col("label_known_in_split").cast("int")).alias("weeks_with_usable_label"),
     F.sum(F.col("friday_was_holiday").cast("int")).alias("friday_holidays"),
     F.min("anchor_date").alias("first_anchor"), F.max("anchor_date").alias("last_anchor"))
 .orderBy("first_anchor").show(truncate=False))

print("Rows per split (date coverage, size, class balance):")
(model_rows.groupBy("split")
 .agg(F.min("date").alias("first_date"), F.max("date").alias("last_date"),
      F.count("*").alias("rows"), F.countDistinct("Name").alias("series"),
      F.round(F.avg("label"), 3).alias("share_up"))
 .orderBy("first_date").show(truncate=False))

print("GDELT window rows (same-rows subset for the price-only vs price+GDELT comparison):")
(model_rows.filter("in_gdelt_window").groupBy("split")
 .agg(F.min("date").alias("first_date"), F.max("date").alias("last_date"),
      F.count("*").alias("rows"), F.round(F.avg("label"), 3).alias("share_up"))
 .orderBy("first_date").show(truncate=False))

print("Exclusions by split and reason:")
excluded.groupBy("split", "exclusion_reason").count().orderBy("split", "exclusion_reason").show(truncate=False)

# gap audit: genuine holes inside a series are kept as gaps, never filled
w_g = Window.partitionBy("Name").orderBy("date")
print("Gaps > 5 calendar days inside a series (kept as gaps, never filled):")
(clean.withColumn("prev_date", F.lag("date").over(w_g))
      .withColumn("gap_days", F.datediff("date", "prev_date"))
      .filter("gap_days > 5").select("Name", "prev_date", "date", "gap_days")
      .orderBy("prev_date", "Name").show(30, truncate=False))
print("Weekly rows excluded because their lookback window crosses such a gap (gappy_history):")
(excluded.filter("exclusion_reason = 'gappy_history'")
         .groupBy("split", "Name").count().orderBy("split", "Name").show(20))

# output schema (identical for train / val / test)
print("Output schema of train / val / test:")
train.printSchema()
mark("evidence tables")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Spark version  : 3.5.5-amzn-1
Application ID : application_1794914433132_0004
Run date (UTC) : 2026-11-17 12:07
Python version : 3.9.25

Row-count reconciliation:
  price rows : raw = clean + rejected + out_of_scope      605910 vs    605910   OK
  weekly rows: weekly_all = model_rows + excluded         125809 vs    125809   OK
  model rows : model_rows = train + val + test            121751 vs    121751   OK

Quality flags (rows are kept, not removed):
  zero-volume rows in clean_prices          : 0
  rows with |daily price change| > 50% (check for unadjusted splits): 6
+-----+----------+------+-------------------+
|Name |date      |close |pchg_day           |
+-----+----------+------+-------------------+
|VRTX |2013-04-19|85.6  |0.6190656326839417 |
|DISCK|2014-08-07|39.78 |-0.5102191578428958|
|DISCA|2014-08-07|40.975|-0.5064442303059503|
|NI   |2015-07-02|16.99 |-0.6261826182618262|
|EBAY |2015-07-20|28.57 |-0.5690149343792428|
|AMD  |2016-04-22|3.99  |0.5229007633587786 |
+-----+--

### 1.9 Write outputs

Saves the eight output tables as Parquet-backed Hive tables in `HIVE_DB`, so both Hive and Spark can read them, then reads each table back and checks its row count. Controlled by `WRITE_OUTPUTS`. When `OUT_S3` is set it also writes a second Parquet copy of each table to S3, so the tables outlive the cluster. `mark(...)` prints the elapsed time at each stage.

In [16]:
if WRITE_OUTPUTS:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")
    expected = {"clean_prices": clean_n, "rejected_prices": rej_n, "out_of_scope_prices": oos_n,
                "weekly_all": weekly_n, "excluded_weeks": excl_n,
                "train": split_n["train"], "val": split_n["validation"], "test": split_n["test"]}
    for name, df in [("clean_prices", clean), ("rejected_prices", rejected),
                     ("out_of_scope_prices", out_of_scope), ("weekly_all", weekly),
                     ("excluded_weeks", excluded), ("train", train), ("val", val), ("test", test)]:
        df.write.mode("overwrite").format("parquet").saveAsTable(f"{HIVE_DB}.{name}")
        n = spark.table(f"{HIVE_DB}.{name}").count()
        assert n == expected[name], f"{name}: expected {expected[name]} rows, read back {n}"
        print(f"saved {HIVE_DB}.{name}: {n} rows")
        if OUT_S3:                                   # optional second copy that outlives the cluster
            try:
                spark.table(f"{HIVE_DB}.{name}").write.mode("overwrite").parquet(f"{OUT_S3}/{name}")
                print(f"  copied to {OUT_S3}/{name}")
            except Exception as e:                   # noqa: BLE001  the Hive table is saved either way
                print(f"  WARNING: S3 copy of {name} failed ({type(e).__name__}: {str(e)[:150]}); the Hive table is unaffected")
mark("write outputs")

print("\nStage timings (seconds, this task only):")
prev = 0.0
for label, t in STAGE_TIMES:
    print(f"  {label:38s} {t - prev:8.1f}   (cumulative {t:8.1f})")
    prev = t

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

saved bigdata.clean_prices: 605887 rows
  copied to s3://big-date-team18-bucket/outputs/clean_prices
saved bigdata.rejected_prices: 23 rows
  copied to s3://big-date-team18-bucket/outputs/rejected_prices
saved bigdata.out_of_scope_prices: 0 rows
  copied to s3://big-date-team18-bucket/outputs/out_of_scope_prices
saved bigdata.weekly_all: 125809 rows
  copied to s3://big-date-team18-bucket/outputs/weekly_all
saved bigdata.excluded_weeks: 4058 rows
  copied to s3://big-date-team18-bucket/outputs/excluded_weeks
saved bigdata.train: 70829 rows
  copied to s3://big-date-team18-bucket/outputs/train
saved bigdata.val: 25340 rows
  copied to s3://big-date-team18-bucket/outputs/val
saved bigdata.test: 25582 rows
  copied to s3://big-date-team18-bucket/outputs/test
[   110.1s] write outputs

Stage timings (seconds, this task only):
  read raw prices                            17.0   (cumulative     17.0)
  clean / reject / out-of-scope              11.6   (cumulative     28.6)
  cross-check reje

In [17]:
for name in ["clean_prices", "rejected_prices", "out_of_scope_prices", "weekly_all", "excluded_weeks"]:
    print(name); spark.table(f"{HIVE_DB}.{name}").printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

clean_prices
root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- zero_volume: boolean (nullable = true)

rejected_prices
root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- _corrupt: string (nullable = true)
 |-- src_file: string (nullable = true)
 |-- date_raw: string (nullable = true)
 |-- reason: string (nullable = true)

out_of_scope_prices
root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- Name: string 

### Hand-over: Part 1 to Part 2

Part 2 reads the prepared tables `train`, `val` and `test`. On EMR it reads them from Hive (`HIVE_DB`, default `bigdata`). In a local run, it takes the DataFrames below.


In [18]:
# Hand-over from Part 1 (data preparation) to Part 2 (modelling).
# Part 2 keeps its own names (d_train, d_val, d_test), so it never overwrites Part 1's DataFrames.
try:
    C_PREPARED = {"train": train, "val": val, "test": test}
except NameError:   # Part 1 was skipped in this session
    print("Part 1 has not run in this session: Part 2 reads the Hive tables train / val / test (EMR) or local_tables/ (laptop).")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Part 2 – Task 3 (Spark ML): baseline, experiments, final model and saved PipelineModel

**Inputs:** Hive tables `train`, `val`, `test` in `HIVE_DB` (one row per stock per week, written by the data-preparation part, Task 2).
**Outputs:** metrics and a run summary (printed and saved as text), prediction tables `pred_final_test` and `pred_gdelt_test`, and three saved PipelineModels.

**Prediction task.** For each stock and trading week, predict `label` = 1 if the open on the next anchor date is above the open on this anchor date. Every feature uses data from before the anchor date. Weeks whose outcome is not known inside their split are not in these tables.

**Experiments**
1. `full_price`: price-only features on all weeks (train 2013-2015, validation 2016, test 2017).
2. `gdelt_price_only` and `gdelt_price_gdelt`: the same rows, restricted to weeks with complete GDELT features (`in_gdelt_window`), without and with the five GDELT features.

**Rules**
- Every learned step (scaling and the model) is fitted on the training set only, inside a `Pipeline`.
- Settings are chosen on the validation set with a manual search over three model families (logistic regression, random forest, gradient-boosted trees; 22 settings per experiment). The setting with the highest validation AUC is chosen (ties: lower validation log-loss). AUC does not depend on a cut-off, so it is not fooled by the majority class. No random folds are used, so time order is kept.
- The chosen settings are stored in `FROZEN` before the test set is used. The test set is used once per experiment.
- The baseline predicts the majority class of the training set, on exactly the same rows as the model.
- Every comparison reports accuracy, balanced accuracy (mean of the recall on up weeks and on down weeks; the baseline always scores 0.5) and AUC (0.5 = no ranking skill), because accuracy alone cannot tell a useful model from one that predicts the majority class for every row.

### 2.0 Config

`ON_EMR` is detected from the Spark master (YARN means the cluster), set it by hand to force a mode. `D_SOURCE` says where the prepared tables come from: the Hive tables `train`, `val`, `test` in `HIVE_DB` on EMR, the DataFrames left in the session by the data-preparation part (combined notebook, local run), or local CSV exports in `local_tables/` (stand-alone dry run). `OUT_S3` (default: the team bucket) makes Part 2 copy its predictions, models and run summary to S3 on EMR as well; set it to None for timed runs. `OUT_SUFFIX` is added to every output name, so each repeated run writes to fresh locations. On EMR, the `%%configure` cell at the top of the notebook must be run first (it is the reference allocation: 2 executors x 2 cores x 6 GiB heap, 2 GiB overhead, 2 GiB driver, dynamic allocation off).


In [19]:
# This EMR image has no numpy for Python: install it for this Spark session (driver and executors).
sc.install_pypi_package("numpy==1.26.4")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.2.3 requires tzdata>=2022.7, which is not installed.
matplotlib 3.9.4 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.9.4 requires cycler>=0.10, which is not installed.
matplotlib 3.9.4 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.9.4 requires importlib-resources>=3.2.0; python_version < "3.10", which is not installed.
matplotlib 3.9.4 requires kiwisolver>=1.3.1, which is not installed.
matplotlib 3.9.4 requires pillow>=8, which is not installed.
pandas 2.2.3 requires python-dateutil>=2.8.2, but you have python-dateutil 2.8.1 which is incompatible.

In [20]:
import json, os, time, itertools, random, numpy
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

spark = SparkSession.builder.getOrCreate()      # in the combined notebook this is the session of Part 1
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "16")

ON_EMR     = None           # None = detect: True when Spark runs on YARN (the cluster), False on a laptop. Set True / False to force.
D_SOURCE   = "auto"         # where the prepared tables come from:
                            #   "hive"    train / val / test in HIVE_DB (EMR; Part 1 must have saved them, WRITE_OUTPUTS = True)
                            #   "session" the DataFrames Part 1 left in this session (combined notebook, local run)
                            #   "csv"     local_tables/train, val, test (stand-alone dry run, see make_synthetic_tables.py)
                            #   "auto"    hive on EMR; otherwise session if Part 1 ran in this notebook, else csv
SEED       = 42             # random seed used by every model and by the bootstrap
OUT_SUFFIX = ""             # new value for each repeated run, e.g. "_run2" (fresh output locations)
FINAL_MODEL = "full_price"  # the final model: one of "full_price", "gdelt_price_only", "gdelt_price_gdelt".
                            # Declared BEFORE any test result is seen. The default is the price-only model on all weeks (largest sample);
                            # change it only from the validation results, never from the test results.

HIVE_DB    = globals().get("HIVE_DB", "bigdata")   # Hive database of the prepared tables (Part 1 sets it in the combined notebook)
OUT_S3     = globals().get("OUT_S3", "s3://big-date-team18-bucket/outputs")   # optional second copy on S3 (EMR only); None = no copy

if ON_EMR is None:
    ON_EMR = spark.sparkContext.master.startswith("yarn")
if D_SOURCE == "auto":
    D_SOURCE = "hive" if ON_EMR else ("session" if "C_PREPARED" in globals() else "csv")
assert D_SOURCE in ("hive", "session", "csv"), f"D_SOURCE must be hive, session, csv or auto, not {D_SOURCE!r}"
assert D_SOURCE != "session" or "C_PREPARED" in globals(), "D_SOURCE = 'session' needs Part 1 (data preparation) to have run first in this notebook"

LOCAL_DIR    = "local_tables"                                   # local CSV exports of train / val / test
LOCAL_OUT    = "local_out"                                      # local prediction tables
EVIDENCE_DIR = "hdfs:///project/evidence/task3" if ON_EMR else "evidence"
MODEL_ROOT   = "hdfs:///project/models"         if ON_EMR else "models"
TOL          = 1e-9                                             # tolerance used by the reload check

RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
_D_T0, D_STAGE_TIMES = time.time(), []
def d_mark(label):
    D_STAGE_TIMES.append((label, round(time.time() - _D_T0, 1)))
    print(f"[{D_STAGE_TIMES[-1][1]:>8.1f}s] {label}")

def s3_copy(label, action):
    """Optional second copy on S3 (EMR only). A failure is reported but never stops the run."""
    if not (ON_EMR and OUT_S3):
        return
    try:
        action()
        print(f"  copied {label} to S3")
    except Exception as e:   # noqa: BLE001  the Hive / HDFS copy is saved either way
        print(f"  WARNING: S3 copy of {label} failed ({type(e).__name__}: {str(e)[:150]}); the Hive / HDFS copy is unaffected")

print("Run ID         :", RUN_ID)
print("Spark version  :", spark.version)
print("Application ID :", spark.sparkContext.applicationId)
print("ON_EMR =", ON_EMR, "| D_SOURCE =", D_SOURCE, "| HIVE_DB =", HIVE_DB, "| SEED =", SEED, "| OUT_SUFFIX =", repr(OUT_SUFFIX))
print("S3 copy        :", OUT_S3 if (ON_EMR and OUT_S3) else "off")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Run ID         : 20261117T120821Z
Spark version  : 3.5.5-amzn-1
Application ID : application_1794914433132_0004
ON_EMR = True | D_SOURCE = hive | HIVE_DB = bigdata | SEED = 42 | OUT_SUFFIX = ''
S3 copy        : s3://big-date-team18-bucket/outputs

### 2.1 Read the prepared tables and check them

The feature lists are the ones defined in the data-preparation notebook. The checks stop the run if a table has the wrong split, a missing target or an unexpected null.

In [21]:
_C_PRICE, _C_GDELT, _C_NOT = (globals().get(n) for n in ("FEATURE_COLS", "GDELT_COLS", "NOT_FEATURES"))   # defined by Part 1, if it ran here
PRICE_COLS = ["pchg_1d", "pchg_5d", "pchg_20d", "vol_ratio_5_20", "volatility_20d", "range_prev"]
GDELT_COLS = ["g_events_7d", "g_events_chg", "g_tone_7d", "g_neg_share_7d", "g_conflict_share_7d"]
# identifiers and target-related columns: never features
NOT_FEATURES = {"Name", "date", "week_friday", "open", "open_next_fri", "label", "pchg_next", "pchg_day",
                "split", "next_anchor_date", "in_gdelt_window", "n_prior", "hist_span_days"}
if _C_PRICE is not None:   # combined notebook: the lists below must be the ones Part 1 used to build the tables
    assert PRICE_COLS == _C_PRICE and GDELT_COLS == _C_GDELT and NOT_FEATURES == _C_NOT, "feature lists differ from Part 1 (data preparation)"
assert not set(PRICE_COLS + GDELT_COLS) & NOT_FEATURES, "an identifier / target column is used as a feature"

SCHEMA = T.StructType([
    T.StructField("Name", T.StringType()), T.StructField("date", T.DateType()),
    T.StructField("week_friday", T.DateType()), T.StructField("open", T.DoubleType()),
    T.StructField("open_next_fri", T.DoubleType()), T.StructField("label", T.IntegerType()),
    T.StructField("pchg_next", T.DoubleType()), T.StructField("split", T.StringType()),
    T.StructField("next_anchor_date", T.DateType()), T.StructField("in_gdelt_window", T.BooleanType()),
    T.StructField("pchg_1d", T.DoubleType()), T.StructField("pchg_5d", T.DoubleType()),
    T.StructField("pchg_20d", T.DoubleType()), T.StructField("vol_ratio_5_20", T.DoubleType()),
    T.StructField("volatility_20d", T.DoubleType()), T.StructField("range_prev", T.DoubleType()),
    T.StructField("g_events_7d", T.LongType()), T.StructField("g_events_chg", T.DoubleType()),
    T.StructField("g_tone_7d", T.DoubleType()), T.StructField("g_neg_share_7d", T.DoubleType()),
    T.StructField("g_conflict_share_7d", T.DoubleType())])

def load(name):
    if D_SOURCE == "csv":   # local CSV export (a single file or a folder written by Spark)
        # FAILFAST: a value that does not fit the schema stops the run instead of silently becoming null
        df = spark.read.option("header", True).option("mode", "FAILFAST").schema(SCHEMA).csv(f"{LOCAL_DIR}/{name}")
    else:
        try:
            src = spark.table(f"{HIVE_DB}.{name}") if D_SOURCE == "hive" else C_PREPARED[name]
        except Exception as e:   # noqa: BLE001
            raise RuntimeError(f"{HIVE_DB}.{name} could not be read ({type(e).__name__}). In the combined notebook, set "
                               f"WRITE_OUTPUTS = True in the Part 1 config and run the last cell of Part 1 first.") from e
        # only the columns used here, with fixed types (a missing column stops the run)
        df = src.select([F.col(f.name).cast(f.dataType).alias(f.name) for f in SCHEMA.fields])
    return df.withColumn("label", F.col("label").cast("double")).cache()

d_train, d_val, d_test = load("train"), load("val"), load("test")
SPLIT_NAME = {"train": "train", "val": "validation", "test": "test"}

for name, df in [("train", d_train), ("val", d_val), ("test", d_test)]:
    missing = set(PRICE_COLS + GDELT_COLS + ["Name", "date", "label", "split", "in_gdelt_window"]) - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"
    assert df.filter(F.col("split") != SPLIT_NAME[name]).count() == 0, f"{name}: rows from another split"
    assert df.filter(F.col("label").isNull()).count() == 0, f"{name}: a row has no target"
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in PRICE_COLS]).first().asDict()
    assert sum(nulls.values()) == 0, f"{name}: null price features {nulls}"
    g_nulls = (df.filter("in_gdelt_window")
                 .select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in GDELT_COLS]).first().asDict())
    assert all((v or 0) == 0 for v in g_nulls.values()), f"{name}: null GDELT features inside the GDELT window {g_nulls}"
    assert df.groupBy("Name", "date").count().filter("count > 1").count() == 0, f"{name}: duplicate (Name, date)"

b = {n: df.agg(F.min("date").alias("lo"), F.max("date").alias("hi")).first() for n, df in
     [("train", d_train), ("val", d_val), ("test", d_test)]}
assert b["train"]["hi"] < b["val"]["lo"] and b["val"]["hi"] < b["test"]["lo"], "splits overlap in time"

print("Rows and date range per split:")
PROFILE = {}
for name, df in [("train", d_train), ("val", d_val), ("test", d_test)]:
    r = df.agg(F.count("*").alias("rows"), F.countDistinct("Name").alias("series"),
               F.avg("label").alias("share_up"),
               F.sum(F.col("in_gdelt_window").cast("int")).alias("gdelt_rows")).first()
    PROFILE[name] = {"rows": r["rows"], "series": r["series"], "share_up": round(r["share_up"], 4),
                     "gdelt_rows": r["gdelt_rows"], "first_date": str(b[name]["lo"]), "last_date": str(b[name]["hi"])}
    print(f"  {name:5s} {PROFILE[name]}")
d_mark("read and check tables")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Rows and date range per split:
  train {'rows': 70829, 'series': 495, 'share_up': 0.5363, 'gdelt_rows': 11811, 'first_date': '2013-03-15', 'last_date': '2015-12-24'}
  val   {'rows': 25340, 'series': 498, 'share_up': 0.5394, 'gdelt_rows': 25340, 'first_date': '2016-01-08', 'last_date': '2016-12-23'}
  test  {'rows': 25582, 'series': 504, 'share_up': 0.5639, 'gdelt_rows': 13005, 'first_date': '2017-01-06', 'last_date': '2017-12-22'}
[     5.7s] read and check tables

### 2.2 Helpers: metrics, pipeline, tuning, weekly bootstrap

Metrics are computed with Spark aggregations (no collection of the data): accuracy, balanced accuracy, AUC, log-loss, Brier score and the confusion matrix. `weekly_bootstrap_diff` resamples whole weeks, because stocks in the same week share market conditions and GDELT features, so rows are not independent. Only about 26 weekly counts are collected.

In [22]:
def with_prob(pred):
    return pred.withColumn("prob_up", vector_to_array("probability")[1])

def rates(tp, fp, fn, tn):
    """Rates from a confusion matrix (plain Python). Balanced accuracy = mean of the recall on up weeks and on down weeks,
    so a model that predicts one class for every row scores exactly 0.5, whatever the share of up weeks."""
    pr = tp / (tp + fp) if (tp + fp) else None
    rc = tp / (tp + fn) if (tp + fn) else None
    rd = tn / (tn + fp) if (tn + fp) else None
    return {"precision_up": pr, "recall_up": rc, "recall_down": rd,
            "f1_up": 2 * pr * rc / (pr + rc) if pr and rc else None,
            "balanced_accuracy": (rc + rd) / 2 if rc is not None and rd is not None else None}

def metrics(df):
    """df has label, prediction and prob_up."""
    e, p, y = 1e-15, F.col("prob_up"), F.col("label")
    r = df.agg(
        F.count("*").alias("n"),
        F.avg(y).alias("share_up"),
        F.avg((F.col("prediction") == y).cast("double")).alias("accuracy"),
        F.sum(((F.col("prediction") == 1) & (y == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("prediction") == 1) & (y == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("prediction") == 0) & (y == 1)).cast("int")).alias("fn"),
        F.sum(((F.col("prediction") == 0) & (y == 0)).cast("int")).alias("tn"),     # tp, fp, fn, tn = confusion matrix
        F.avg(-(y * F.log(F.greatest(p, F.lit(e))) + (1 - y) * F.log(F.greatest(1 - p, F.lit(e))))).alias("logloss"),
        F.avg((p - y) * (p - y)).alias("brier")).first().asDict()
    r.update(rates(r["tp"], r["fp"], r["fn"], r["tn"]))
    r["auc"] = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="prob_up",
                                             metricName="areaUnderROC").evaluate(df)
    return {k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()}

def build_pipeline(algo, feats, p):
    assembler = VectorAssembler(inputCols=feats, outputCol="features_raw", handleInvalid="error")
    scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
    if algo == "lr":
        clf = LogisticRegression(featuresCol="features", labelCol="label", maxIter=100,
                                 standardization=False, **p)
    elif algo == "rf":
        clf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=SEED, **p)
    elif algo == "gbt":
        clf = GBTClassifier(featuresCol="features", labelCol="label", seed=SEED, **p)
    else:
        raise ValueError(algo)
    return Pipeline(stages=[assembler, scaler, clf])   # scaler and model are learned from the training rows only

def expand(grid):
    keys = list(grid)
    return [dict(zip(keys, vals)) for vals in itertools.product(*[grid[k] for k in keys])]

# The search space: three model families, 22 settings per experiment (the data are small, so every fit takes seconds).
# Only the validation rows are used to choose among them; the test rows are not touched until the choice is frozen.
GRIDS = {
    "lr":  expand({"regParam": [0.001, 0.01, 0.1], "elasticNetParam": [0.0, 0.5, 1.0]})      # regularised logistic regression
            + [{"regParam": 0.0, "elasticNetParam": 0.0}],                                 # and the unregularised one
    "rf":  expand({"numTrees": [100, 200], "maxDepth": [3, 6], "minInstancesPerNode": [20, 100]}),   # shallow trees, big leaves: the data are noisy
    "gbt": expand({"maxIter": [20, 40], "maxDepth": [2, 4], "stepSize": [0.1]})}                    # small boosted trees
SELECTION_RULE = "highest validation AUC; ties: lower validation log-loss"

def weekly_bootstrap_diff(weeks, n_boot=2000, seed=SEED):
    """weeks: list of (n_rows, correct_a, correct_b) per week.
    Returns the accuracy difference (b - a) and a 95% interval from resampling whole weeks."""
    rng, k = random.Random(seed), len(weeks)
    def diff(sample):
        n = sum(w[0] for w in sample)
        return (sum(w[2] for w in sample) - sum(w[1] for w in sample)) / n
    draws = sorted(diff([weeks[rng.randrange(k)] for _ in range(k)]) for _ in range(n_boot))
    return {"diff": round(diff(weeks), 6), "ci95": [round(draws[int(0.025 * n_boot)], 6),
            round(draws[int(0.975 * n_boot) - 1], 6)], "n_weeks": k, "n_boot": n_boot}

def weekly_counts(pred_a, pred_b):
    """Per-week row count and correct predictions of two prediction frames on the same rows."""
    a = pred_a.select("Name", "date", (F.col("prediction") == F.col("label")).cast("int").alias("ok_a"))
    b = pred_b.select("Name", "date", (F.col("prediction") == F.col("label")).cast("int").alias("ok_b"))
    rows = (a.join(b, ["Name", "date"]).groupBy("date")
             .agg(F.count("*").alias("n"), F.sum("ok_a").alias("ok_a"), F.sum("ok_b").alias("ok_b"))
             .orderBy("date").collect())
    return [(r["n"], r["ok_a"], r["ok_b"]) for r in rows]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### 2.3 Experiments

For each experiment: search all settings in `GRIDS` on validation (highest validation AUC wins), store the choice in `FROZEN`, refit the chosen pipeline on the training rows, and only then predict the test rows. The baseline uses the training-set majority class and the training share of up weeks as its probability.

In [23]:
FROZEN, RESULTS, PREDS, MODELS, VAL_TABLES = {}, {}, {}, {}, {}

def run_experiment(name, feats, tr, va, te):
    tr, va = tr.cache(), va.cache()
    share = tr.agg(F.avg("label")).first()[0]                    # baseline: training majority class
    majority = 1.0 if share >= 0.5 else 0.0
    table = []
    for algo, settings in GRIDS.items():
        for p in settings:
            m = build_pipeline(algo, feats, p).fit(tr)
            v = metrics(with_prob(m.transform(va)))
            table.append({"algo": algo, "params": p, "val_auc": v["auc"], "val_accuracy": v["accuracy"],
                          "val_balanced_accuracy": v["balanced_accuracy"], "val_logloss": v["logloss"]})
            print(f"  [{name}] {algo} {p}: val AUC {v['auc']:.4f}  acc {v['accuracy']:.4f}  bal.acc {v['balanced_accuracy']:.4f}  logloss {v['logloss']:.4f}")
    best = max(table, key=lambda r: (r["val_auc"], -r["val_logloss"]))           # SELECTION_RULE; validation rows only
    for algo in GRIDS:                                                             # the best setting of each family, for the report
        top = max((r for r in table if r["algo"] == algo), key=lambda r: (r["val_auc"], -r["val_logloss"]))
        print(f"  [{name}] best {algo}: val AUC {top['val_auc']:.4f}  {top['params']}")
    FROZEN[name] = {"features": feats, "algo": best["algo"], "params": best["params"]}   # frozen BEFORE the test set is used
    print(f"  [{name}] FROZEN: {FROZEN[name]}")
    model = build_pipeline(best["algo"], feats, best["params"]).fit(tr)
    val_pred = with_prob(model.transform(va))
    test_pred = with_prob(model.transform(te)).cache()           # first and only use of the test rows in this experiment
    base = lambda df: df.withColumn("prediction", F.lit(majority)).withColumn("prob_up", F.lit(float(share)))
    RESULTS[name] = {"rows": {"train": tr.count(), "val": va.count(), "test": te.count()},
                     "baseline": {"majority_class": majority, "train_share_up": round(share, 4),
                                  "val": metrics(base(va)), "test": metrics(base(te))},
                     "model": {"val": metrics(val_pred), "test": metrics(test_pred)}}
    PREDS[name], MODELS[name], VAL_TABLES[name] = test_pred, model, table
    r = RESULTS[name]
    bt, mt = r["baseline"]["test"], r["model"]["test"]
    print(f"  [{name}] TEST  baseline: acc {bt['accuracy']:.4f}  bal.acc {bt['balanced_accuracy']:.4f}  AUC {bt['auc']:.4f}")
    print(f"  [{name}] TEST  model   : acc {mt['accuracy']:.4f}  bal.acc {mt['balanced_accuracy']:.4f}  AUC {mt['auc']:.4f}  logloss {mt['logloss']:.4f}")

# Experiment 1: price-only, all weeks
run_experiment("full_price", PRICE_COLS, d_train, d_val, d_test)
d_mark("experiment full_price")

# Experiment 2: the same rows with and without GDELT features (only weeks with complete GDELT windows)
tr_g, va_g, te_g = [df.filter("in_gdelt_window") for df in (d_train, d_val, d_test)]
run_experiment("gdelt_price_only", PRICE_COLS, tr_g, va_g, te_g)
run_experiment("gdelt_price_gdelt", PRICE_COLS + GDELT_COLS, tr_g, va_g, te_g)
n_a, n_b = PREDS["gdelt_price_only"].count(), PREDS["gdelt_price_gdelt"].count()
n_j = PREDS["gdelt_price_only"].select("Name", "date").join(PREDS["gdelt_price_gdelt"].select("Name", "date"), ["Name", "date"]).count()
assert n_a == n_b == n_j, "the two GDELT-experiment models were not scored on the same rows"
assert FINAL_MODEL in PREDS, f"FINAL_MODEL must be one of {list(PREDS)}"
print("FINAL_MODEL (declared before the test results):", FINAL_MODEL, "->", FROZEN[FINAL_MODEL])
d_mark("experiments gdelt_price_only / gdelt_price_gdelt")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

  [full_price] lr {'regParam': 0.001, 'elasticNetParam': 0.0}: val AUC 0.5197  acc 0.5387  bal.acc 0.4997  logloss 0.6897
  [full_price] lr {'regParam': 0.001, 'elasticNetParam': 0.5}: val AUC 0.5200  acc 0.5389  bal.acc 0.4998  logloss 0.6896
  [full_price] lr {'regParam': 0.001, 'elasticNetParam': 1.0}: val AUC 0.5203  acc 0.5389  bal.acc 0.4997  logloss 0.6896
  [full_price] lr {'regParam': 0.01, 'elasticNetParam': 0.0}: val AUC 0.5200  acc 0.5388  bal.acc 0.4998  logloss 0.6896
  [full_price] lr {'regParam': 0.01, 'elasticNetParam': 0.5}: val AUC 0.5218  acc 0.5393  bal.acc 0.4999  logloss 0.6896
  [full_price] lr {'regParam': 0.01, 'elasticNetParam': 1.0}: val AUC 0.5281  acc 0.5394  bal.acc 0.5000  logloss 0.6900
  [full_price] lr {'regParam': 0.1, 'elasticNetParam': 0.0}: val AUC 0.5212  acc 0.5393  bal.acc 0.5001  logloss 0.6896
  [full_price] lr {'regParam': 0.1, 'elasticNetParam': 0.5}: val AUC 0.5000  acc 0.5394  bal.acc 0.5000  logloss 0.6901
  [full_price] lr {'regParam': 

### 2.4 Does GDELT help? Same rows, weekly bootstrap

GDELT features are market wide, so every stock in a week shares the same values. The effective number of independent GDELT observations is the number of weeks, not the number of rows, so the interval below resamples whole weeks.

In [24]:
COMPARE = {}
wk = weekly_counts(PREDS["gdelt_price_only"], PREDS["gdelt_price_gdelt"])
COMPARE["accuracy_gain_price_plus_gdelt_vs_price_only"] = weekly_bootstrap_diff(wk)

# the final model against the majority-class baseline, same test rows
fp = PREDS[FINAL_MODEL]
base_pred = fp.withColumn("prediction", F.lit(RESULTS[FINAL_MODEL]["baseline"]["majority_class"]))
wk2 = weekly_counts(base_pred, fp)
COMPARE[f"accuracy_gain_{FINAL_MODEL}_model_vs_baseline"] = weekly_bootstrap_diff(wk2)

print(json.dumps(COMPARE, indent=2))
g, p = RESULTS["gdelt_price_gdelt"]["model"]["test"], RESULTS["gdelt_price_only"]["model"]["test"]
print(f"Same test rows ({g['n']}): price-only acc {p['accuracy']:.4f} bal.acc {p['balanced_accuracy']:.4f} AUC {p['auc']:.4f}  |  "
      f"price+GDELT acc {g['accuracy']:.4f} bal.acc {g['balanced_accuracy']:.4f} AUC {g['auc']:.4f}")
d_mark("GDELT comparison")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{
  "accuracy_gain_price_plus_gdelt_vs_price_only": {
    "diff": 0.026836,
    "ci95": [
      -0.07561,
      0.128209
    ],
    "n_weeks": 26,
    "n_boot": 2000
  },
  "accuracy_gain_full_price_model_vs_baseline": {
    "diff": 0.0,
    "ci95": [
      0.0,
      0.0
    ],
    "n_weeks": 51,
    "n_boot": 2000
  }
}
Same test rows (13005): price-only acc 0.4607 bal.acc 0.4975 AUC 0.4795  |  price+GDELT acc 0.4875 bal.acc 0.4737 AUC 0.4439
[   265.2s] GDELT comparison

### 2.5 Where is the final model reliable? Test period by quarter and volatility group

Volatility groups use the terciles of `volatility_20d` in the **training** rows. The baseline is the same majority-class rule on the same rows. Per-slice counts are kept so that small slices are visible.

In [25]:
q1, q2 = d_train.approxQuantile("volatility_20d", [1 / 3, 2 / 3], 0.001)
vol_group = (F.when(F.col("volatility_20d") <= q1, "1_low").when(F.col("volatility_20d") <= q2, "2_mid").otherwise("3_high"))
maj = RESULTS[FINAL_MODEL]["baseline"]["majority_class"]
slices = (PREDS[FINAL_MODEL].withColumn("vol_group", vol_group)
          .withColumn("period", F.concat(F.year("date").cast("string"), F.lit("Q"), F.quarter("date").cast("string")))
          .groupBy("period", "vol_group")
          .agg(F.count("*").alias("n"), F.round(F.avg("label"), 3).alias("share_up"),
               F.round(F.avg((F.col("prediction") == F.col("label")).cast("double")), 4).alias("acc_model"),
               F.round(F.avg((F.lit(maj) == F.col("label")).cast("double")), 4).alias("acc_baseline"))
          .withColumn("lift", F.round(F.col("acc_model") - F.col("acc_baseline"), 4))
          .orderBy("period", "vol_group"))
RELIABILITY = [r.asDict() for r in slices.collect()]          # 12 aggregate rows only
slices.show(20, truncate=False)
print("vol_group cut points (training terciles):", round(q1, 6), round(q2, 6))
d_mark("reliability slices")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+---------+----+--------+---------+------------+----+
|period|vol_group|n   |share_up|acc_model|acc_baseline|lift|
+------+---------+----+--------+---------+------------+----+
|2017Q1|1_low    |3109|0.563   |0.5629   |0.5629      |0.0 |
|2017Q1|2_mid    |1859|0.559   |0.5594   |0.5594      |0.0 |
|2017Q1|3_high   |1528|0.526   |0.5262   |0.5262      |0.0 |
|2017Q2|1_low    |3399|0.548   |0.5475   |0.5475      |0.0 |
|2017Q2|2_mid    |1680|0.514   |0.5137   |0.5137      |0.0 |
|2017Q2|3_high   |1430|0.511   |0.5112   |0.5112      |0.0 |
|2017Q3|1_low    |2980|0.577   |0.5772   |0.5772      |0.0 |
|2017Q3|2_mid    |1893|0.606   |0.6064   |0.6064      |0.0 |
|2017Q3|3_high   |1656|0.586   |0.5857   |0.5857      |0.0 |
|2017Q4|1_low    |2743|0.58    |0.58     |0.58        |0.0 |
|2017Q4|2_mid    |1599|0.602   |0.6016   |0.6016      |0.0 |
|2017Q4|3_high   |1706|0.577   |0.5774   |0.5774      |0.0 |
+------+---------+----+--------+---------+------------+----+

vol_group cut points (t

### 2.6 Write predictions and save the PipelineModels

Prediction tables use `OUT_SUFFIX`, so repeated runs do not overwrite each other. The complete PipelineModel (assembler, scaler and classifier) is saved. Copy `MODEL_ROOT` to S3 before the cluster is terminated, because HDFS is erased with the cluster.

In [26]:
def write_table(df, name):
    full = f"{name}{OUT_SUFFIX}"
    if ON_EMR:
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")
        df.write.mode("overwrite").format("parquet").saveAsTable(f"{HIVE_DB}.{full}")
        n = spark.table(f"{HIVE_DB}.{full}").count()
    else:
        df.write.mode("overwrite").parquet(f"{LOCAL_OUT}/{full}")
        n = spark.read.parquet(f"{LOCAL_OUT}/{full}").count()
    assert n == df.count(), f"{full}: read back {n} rows"
    s3_copy(f"table {full}", lambda: spark.table(f"{HIVE_DB}.{full}").write.mode("overwrite").parquet(f"{OUT_S3}/{full}"))
    print(f"saved {full}: {n} rows")
    return full

keep = ["Name", "date", "split", "label", "pchg_next", "prediction", "prob_up", "volatility_20d", "in_gdelt_window"]
out_final = (PREDS[FINAL_MODEL].withColumn("model_id", F.lit(FINAL_MODEL)).select(*keep, "model_id"))   # reload_check reads model_id
out_gdelt = (PREDS["gdelt_price_only"].select("Name", "date", F.col("prediction").alias("pred_price_only"),
                                              F.col("prob_up").alias("prob_up_price_only"))
             .join(PREDS["gdelt_price_gdelt"].select("Name", "date", "label", "pchg_next",
                   F.col("prediction").alias("pred_price_gdelt"), F.col("prob_up").alias("prob_up_price_gdelt")),
                   ["Name", "date"]))
TABLES = {"pred_final_test": write_table(out_final, "pred_final_test"),
          "pred_gdelt_test": write_table(out_gdelt, "pred_gdelt_test")}

MODEL_PATHS = {}
for name, model in MODELS.items():
    MODEL_PATHS[name] = f"{MODEL_ROOT}/{name}{OUT_SUFFIX}"
    model.write().overwrite().save(MODEL_PATHS[name])
    print("saved PipelineModel:", MODEL_PATHS[name])
    s3_copy(f"model {name}", lambda m=model, n=name: m.write().overwrite().save(f"{OUT_S3}/models/{n}{OUT_SUFFIX}"))
if not ON_EMR and D_SOURCE == "session":   # reload_check.py reads the local test table from local_tables/test
    (d_test.select(*SCHEMA.names).coalesce(1).write.mode("overwrite").option("header", True).csv(f"{LOCAL_DIR}/test"))
    print(f"exported {LOCAL_DIR}/test for reload_check.py")
d_mark("write predictions and save models")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

  copied table pred_final_test to S3
saved pred_final_test: 25582 rows
  copied table pred_gdelt_test to S3
saved pred_gdelt_test: 13005 rows
saved PipelineModel: hdfs:///project/models/full_price
  copied model full_price to S3
saved PipelineModel: hdfs:///project/models/gdelt_price_only
  copied model gdelt_price_only to S3
saved PipelineModel: hdfs:///project/models/gdelt_price_gdelt
  copied model gdelt_price_gdelt to S3
[   281.0s] write predictions and save models

### 2.7 Run summary (text evidence)

Everything the setup guide asks for in one record: software versions, seeds, data cut-offs, chosen settings, metrics and where the saved models are. It is printed here and saved as text. The fresh-session reload check (`scripts/reload_check.py`) reads the model and prediction locations from this run.

In [27]:
def save_text(name, text):
    if ON_EMR:   # a folder with one part file
        df_txt = spark.createDataFrame([(text,)], ["line"]).coalesce(1)
        df_txt.write.mode("overwrite").text(f"{EVIDENCE_DIR}/{name}")
        s3_copy(f"evidence {name}", lambda: df_txt.write.mode("overwrite").text(f"{OUT_S3}/evidence/{name}"))
    else:
        os.makedirs(EVIDENCE_DIR, exist_ok=True)
        with open(f"{EVIDENCE_DIR}/{name}", "w") as f:
            f.write(text)

prev, timings = 0.0, []
for label, t in D_STAGE_TIMES:
    timings.append({"stage": label, "seconds": round(t - prev, 1), "cumulative": t})
    prev = t

SUMMARY = {
    "run_id": RUN_ID, "spark_version": spark.version, "application_id": spark.sparkContext.applicationId,
    "on_emr": ON_EMR, "hive_db": HIVE_DB, "s3_copy": (OUT_S3 if ON_EMR else None), "data_source": D_SOURCE, "seed": SEED, "out_suffix": OUT_SUFFIX, "final_model": FINAL_MODEL,
    "task": "classify whether the next anchor-date open is above this anchor-date open",
    "features_price": PRICE_COLS, "features_gdelt": GDELT_COLS,
    "split_profile": PROFILE, "search_space": GRIDS, "selection_rule": SELECTION_RULE,
    "frozen_choices": FROZEN, "results": RESULTS, "comparisons": COMPARE,
    "reliability_by_quarter_and_volatility": RELIABILITY,
    "validation_search": VAL_TABLES, "model_paths": MODEL_PATHS, "prediction_tables": TABLES,
    "reload_tolerance": TOL, "stage_timings": timings}
text = json.dumps(SUMMARY, indent=2, default=str)
print(text)
save_text(f"model_run_summary{OUT_SUFFIX}.json", text)
d_mark("run summary")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{
  "run_id": "20261117T120821Z",
  "spark_version": "3.5.5-amzn-1",
  "application_id": "application_1794914433132_0004",
  "on_emr": true,
  "hive_db": "bigdata",
  "s3_copy": "s3://big-date-team18-bucket/outputs",
  "data_source": "hive",
  "seed": 42,
  "out_suffix": "",
  "final_model": "full_price",
  "task": "classify whether the next anchor-date open is above this anchor-date open",
  "features_price": [
    "pchg_1d",
    "pchg_5d",
    "pchg_20d",
    "vol_ratio_5_20",
    "volatility_20d",
    "range_prev"
  ],
  "features_gdelt": [
    "g_events_7d",
    "g_events_chg",
    "g_tone_7d",
    "g_neg_share_7d",
    "g_conflict_share_7d"
  ],
  "split_profile": {
    "train": {
      "rows": 70829,
      "series": 495,
      "share_up": 0.5363,
      "gdelt_rows": 11811,
      "first_date": "2013-03-15",
      "last_date": "2015-12-24"
    },
    "val": {
      "rows": 25340,
      "series": 498,
      "share_up": 0.5394,
      "gdelt_rows": 25340,
      "first_date": "2016-01-